# 01. KIC 13F 표본 추출 (파일럿 P1)

`experiment_protocol.md` v1.0의 **2절 표본 구성 규칙**과 **1절 시점 규칙**만 구현한다.

| 표본 | 원천 | 10-K 적격성 기준일 | SIC |
|---|---|---|---|
| S1 대표 표본 | KIC 13F 2026-06-30 | 13F 접수일 이전 | 현재 SIC |
| S3 사건 표본 | KIC 13F 2022-06-30 | 2022-10-07 이전 (당일 제외) | 당시 공시 헤더의 `ASSIGNED-SIC` |

**실행 전 확인**
- SEC는 연락처가 담긴 User-Agent를 요구한다. 아래 `USER_AGENT`에 본인 영문 이름과 이메일을 넣는다.
- 처리 규칙을 바꾸고 싶다면 코드를 고치기 전에 프로토콜을 v1.1로 개정한다.
- `USE_FIXTURES = True`로 두면 네트워크 없이 모의 응답으로 전체 흐름을 점검할 수 있다 (실제 KIC 데이터 아님).

In [12]:
# ── 설정 ─────────────────────────────────────────────
USE_FIXTURES = False   # True: 모의 응답으로 스모크 테스트 (실제 데이터 아님)
USER_AGENT = "Blind Spot pilot Hyeseong Kim energy_hs@naver.com"   # ← 반드시 수정 (ASCII만)

KIC_CIK = 1441689
SAMPLE_SIZE = 12

In [13]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "tests")]
import kic13f as k

if USE_FIXTURES:
    from fixtures import FakeClient
    client = FakeClient()
    OUT_DIR = ROOT / "data" / "processed_fixture"
    print("⚠ 모의 응답 모드: 결과는 실제 KIC 보유내역이 아니다.")
else:
    client = k.SecClient(USER_AGENT, cache_dir=ROOT / "data" / "raw" / "sec_cache")
    OUT_DIR = ROOT / "data" / "processed"

SPECS = {
    "S1": k.SampleSpec("S1", "2026-06-30", cutoff=None, use_historical_sic=False, size=SAMPLE_SIZE),
    "S3": k.SampleSpec("S3", "2022-06-30", cutoff=date(2022, 10, 7), use_historical_sic=True, size=SAMPLE_SIZE),
}

## 1. CUSIP → CIK 수작업 대응표

이름 대조로 연결되지 않은 증권(상태 `failed`, `review`)은 `data/manual/cusip_cik_manual.csv`에 직접 추가한다.
- 열: `cusip, cik, note`
- `review` 후보는 자동 확정하지 않는다. EDGAR에서 회사를 확인한 뒤에만 이 파일에 넣는다.
- 이 파일의 변경 이력도 Git으로 남긴다.

In [14]:
MANUAL_PATH = ROOT / "data" / "manual" / ("cusip_cik_manual_fixture.csv" if USE_FIXTURES else "cusip_cik_manual.csv")
if MANUAL_PATH.exists():
    manual = pd.read_csv(MANUAL_PATH, dtype={"cusip": str})
else:
    manual = pd.DataFrame(columns=["cusip", "cik", "note"])
print(f"수작업 매핑 {len(manual)}건 ← {MANUAL_PATH.relative_to(ROOT)}")

tickers = k.load_ticker_table(client)
print(f"EDGAR 기업 목록 {len(tickers):,}건")

수작업 매핑 2건 ← data\manual\cusip_cik_manual.csv
EDGAR 기업 목록 10,445건


## 2. 표본 구성

In [15]:
results = {}
for name, spec in SPECS.items():
    res = k.build_sample(client, KIC_CIK, spec, tickers, manual)
    results[name] = res
    m = res.metadata
    print(f"[{name}] 13F {m['accession']} | 기준일 {m['report_date']} | 접수일 {m['filing_date']} "
          f"| value 단위 x{m['value_multiplier']} | 정정공시 미사용 {m['amendments_ignored']}")
    print(f"      매핑 상태 {m['mapping_status_counts']}")

HTTPError: 403 Client Error: Forbidden for url: https://data.sec.gov/submissions/CIK0001100663.json

In [ ]:
cols = ["rank", "cik", "edgar_name", "tickers", "sic", "value_usd", "weight", "latest_10k_filing_date"]
results["S1"].sample[cols].style.format({"value_usd": "{:,.0f}", "weight": "{:.2%}"})

In [ ]:
results["S3"].sample[cols].style.format({"value_usd": "{:,.0f}", "weight": "{:.2%}"})

## 3. 제외 내역과 매핑 실패

In [ ]:
for name, res in results.items():
    print(f"[{name}] 제외 {len(res.excluded)}건")
    display(res.excluded.sort_values("value_usd", ascending=False).head(15))

In [ ]:
for name, res in results.items():
    print(f"[{name}] 미확정 매핑 상위 (경계값 {res.p1['boundary_value_usd']:,.0f})")
    display(res.mapping_failures.head(15))

## 4. P1 판정

성공 기준 (프로토콜 7.1): S1·S3 모두 상위 12개 선정 완료, 그리고 **12위 경계 이상의 미확정 매핑 0건**.

실패하면 결과를 보고 표본 규칙을 바꾸지 않는다. 미확정 매핑을 수작업 대응표로 해소한 뒤 재실행한다.

In [ ]:
p1 = pd.DataFrame([r.p1 for r in results.values()])
display(p1)
print("P1 전체:", "통과" if p1["pass"].all() else "미통과 → 수작업 대응표 보완 후 재실행")

In [ ]:
for res in results.values():
    path = k.save_result(res, OUT_DIR)
    print("저장:", path.relative_to(ROOT))

## 5. 해석 시 유의점

- 이 표본은 **KIC가 SEC에 공시한 13F 보유내역에서 추출해 비중을 재정규화한 샘플**이며, KIC 전체 포트폴리오를 대표하지 않는다.
- S3 비중은 2022-06-30 기준값이며, 이후 사건일(2022-10-07)까지의 매매는 반영되지 않는다.
- 13F 범위상 미국 상장주식 롱포지션만 포함되고, 10-K 미제출 발행사(ETF, 20-F 제출 해외 발행사)는 규칙에 따라 제외된다.
- 다음 단계(P2): S1 상위 5개 발행사에 대해 사람이 먼저 정답 라벨을 붙인 뒤 공시 노출 추출을 실행한다.